# Preparação e curadoria do dataset SFT

Cria filas balanceadas para Python, PHP, JavaScript e SQL, excluindo o benchmark. O SFT somente é construído depois da aprovação humana. Consulte também o README desta pasta.

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

if Path('/workspace').exists():
    search_root = Path('/workspace')       # RunPod
elif Path('/content').exists():
    search_root = Path('/content')         # Google Colab
else:
    search_root = Path.cwd()               # Execução local

configs = list(search_root.rglob('configs/default.yaml'))

if not configs and search_root == Path('/content'):
    from google.colab import files
    print('Selecione o arquivo ZIP completo do projeto Celx.')
    uploaded = files.upload()
    archives = [Path('/content') / name for name in uploaded if name.lower().endswith('.zip')]
    assert archives, 'Nenhum arquivo ZIP foi enviado.'
    destination = Path('/content/legacy-doc-project')
    destination.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(archives[0]), str(destination))
    configs = list(destination.rglob('configs/default.yaml'))

assert configs, 'configs/default.yaml não foi encontrado no projeto ou no ZIP.'
repo_dir = configs[0].parents[1]
print('Ambiente:', 'Colab' if search_root == Path('/content') else 'RunPod/local')
print('Projeto:', repo_dir)

## Instalar dependências

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '-r', str(repo_dir/'requirements.txt')], check=True)

## Normalizar CodeXGLUE

In [ ]:
subprocess.run([sys.executable, '-m', 'scripts.prepare_dataset'], cwd=repo_dir, check=True)

## Reservar o benchmark de 400 casos

In [ ]:
benchmark = repo_dir/'dataset/benchmark/expanded_100.jsonl'
if not benchmark.exists():
    subprocess.run([sys.executable, '-m', 'scripts.build_benchmark'], cwd=repo_dir, check=True)
assert benchmark.exists()
print('Benchmark reservado:', benchmark)

## Criar fila piloto

O piloto usa 50 exemplos de treino e 10 de validação/teste por linguagem, totalizando 280 exemplos.

In [ ]:
command = [sys.executable, '-m', 'scripts.create_curated_queue',
           '--train-per-language', '50',
           '--validation-per-language', '10',
           '--test-per-language', '10']
subprocess.run(command, cwd=repo_dir, check=True)
manifest_path = repo_dir/'dataset/curated/queues/manifest.json'
print(manifest_path.read_text(encoding='utf-8'))

## Baixar os arquivos produzidos

Esta célula cria um ZIP com as filas, manifestos, templates e, quando disponível, o dataset SFT. No Colab, o download começa automaticamente.

In [ ]:
import zipfile
archive_path = search_root/'etapa_05_dataset_sft_resultados.zip'
include_paths = [
    repo_dir/'dataset/curated',
    repo_dir/'dataset/processed/sft',
    repo_dir/'notebooks/05_preparacao_dataset_sft.ipynb',
    repo_dir/'docs/CURADORIA_SFT.md',
    repo_dir/'docs/CHECKLIST_CURADORIA.md',
]
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for source in include_paths:
        if source.is_file():
            archive.write(source, source.relative_to(repo_dir).as_posix())
        elif source.is_dir():
            for file in source.rglob('*'):
                if file.is_file():
                    archive.write(file, file.relative_to(repo_dir).as_posix())
print('Arquivo criado:', archive_path, '-', round(archive_path.stat().st_size/1024, 1), 'KB')
if search_root == Path('/content'):
    from google.colab import files
    files.download(str(archive_path))
else:
    print('Baixe manualmente:', archive_path)

## Pausa obrigatória para curadoria

Preencha `documentation_pt`, informe o revisor e altere `review_status` para `approved`. Depois salve os registros aprovados em `dataset/curated/train.jsonl`, `validation.jsonl` e `test.jsonl`. Consulte `docs/CURADORIA_SFT.md`.

In [ ]:
curated = repo_dir/'dataset/curated'
for split in ('train', 'validation', 'test'):
    path = curated/f'{split}.jsonl'
    print(split, 'PRONTO' if path.exists() else 'PENDENTE', path)

## Construir o SFT depois da aprovação

A célula falha se algum exemplo não estiver aprovado, estiver incompleto ou aparecer no benchmark.

In [ ]:
import json
curated = repo_dir/'dataset/curated'
ready = True
for split in ('train', 'validation', 'test'):
    path = curated/f'{split}.jsonl'
    if not path.exists():
        print(f'⏸️ {split}: arquivo curado ainda não existe.')
        ready = False
        continue
    rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    pending = sum(row.get('review_status') != 'approved' for row in rows)
    empty = sum(not str(row.get('documentation_pt', '')).strip() for row in rows)
    print(f'{split}: {len(rows)} exemplos; {pending} sem aprovação; {empty} sem documentação.')
    ready = ready and bool(rows) and pending == 0 and empty == 0

if ready:
    subprocess.run([sys.executable, '-m', 'scripts.build_sft_dataset'], cwd=repo_dir, check=True)
    sft_manifest = repo_dir/'dataset/processed/sft/manifest.json'
    print('✅ Dataset SFT construído:')
    print(sft_manifest.read_text(encoding='utf-8'))
else:
    print('\nCURADORIA PENDENTE: baixe o ZIP das filas, revise os exemplos e depois retorne a esta célula.')